# Jacobian Chain Rule Explorer

Chapter 11: Matrix Calculus · *Mathematics for AI and Machine Learning* by Fuheng Wu

Compose two vector maps and compare direct Jacobian with chain-rule product.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-11/jacobian-chain/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import math

import torch

torch.set_printoptions(precision=5)

# The demo's maps and default point v = (0.6, -0.8)
def f(v):
    return torch.stack([torch.sin(v[0]) + v[1] ** 2, v[0] * v[1]])


def g(u):
    return torch.stack([u[0] ** 2 + u[1], torch.exp(0.2 * u[1])])


v = torch.tensor([0.6, -0.8], dtype=torch.float64)
u = f(v)

# ── Chain rule: J_{g o f}(v) = J_g(f(v)) J_f(v) ─────────────────────────────
Jf = torch.autograd.functional.jacobian(f, v)
Jg = torch.autograd.functional.jacobian(g, u)
J_chain = Jg @ Jf
J_direct = torch.autograd.functional.jacobian(lambda w: g(f(w)), v)
print("J_f(v) =\n", Jf)
print("J_g(f(v)) =\n", Jg)
print("J_g J_f =\n", J_chain)
print("matches autograd of g(f(v)):", torch.allclose(J_chain, J_direct))

# J_f by hand: [[cos x, 2y], [y, x]]
x, y = v.tolist()
print("J_f by hand:", [[math.cos(x), 2 * y], [y, x]])

# ── Finite differences agree to O(eps) ─────────────────────────────────────
eps = 1e-6
J_fd = torch.stack([(g(f(v + eps * e)) - g(f(v))) / eps for e in torch.eye(2, dtype=v.dtype)], dim=1)
print("\nmax |chain - finite difference| =", (J_chain - J_fd).abs().max().item())

# Order matters: J_f J_g is a different matrix
print("J_f J_g == J_g J_f ?", torch.allclose(Jf @ Jg, J_chain))

J_f(v) =
 tensor([[ 0.82534, -1.60000],
        [-0.80000,  0.60000]], dtype=torch.float64)
J_g(f(v)) =
 tensor([[2.40928, 1.00000],
        [0.00000, 0.18169]], dtype=torch.float64)
J_g J_f =
 tensor([[ 1.18847, -3.25486],
        [-0.14535,  0.10902]], dtype=torch.float64)
matches autograd of g(f(v)): True
J_f by hand: [[0.8253356149096783, -1.6], [-0.8, 0.6]]

max |chain - finite difference| = 4.969319772563097e-06
J_f J_g == J_g J_f ? False
